# Bid Optimization Engine

**Role:** AI Algorithm Developer — Home Assignment
**Company:** Feedvisor
**Objective:** Given 30 days of keyword advertising data, recommend one optimized bid per keyword across 10 campaigns to maximize ROAS while respecting hard campaign budget constraints.

---

## Table of Contents
1. [Task Objective & Business Context](#1-task-objective--business-context)
2. [How Business Constraints Shape the Algorithm](#2-how-business-constraints-shape-the-algorithm)
3. [Data Investigation — Questions & EDA](#3-data-investigation--questions--eda)
4. [EDA Findings & Algorithm Decisions](#4-eda-findings--algorithm-decisions)

---
## 1. Task Objective & Business Context

Before writing a single line of code, we need to understand what the business actually needs and what constraints are non-negotiable. These constraints will directly determine the algorithm design.

### 1.1 Task Objective

Feedvisor manages advertising campaigns for brands on Amazon. The optimizer's job is to set a **maximum cost-per-click (CPC) bid** for each keyword — a standing instruction that stays in effect until the next daily cycle.

**Input:** `campaign_data.csv` — 30 days of daily keyword-level performance data (~200 keywords, 10 campaigns).
**Output:** One recommended bid per unique keyword, written to `bid_recommendations.csv`.

The bid is a lever with two opposing forces:
- **Too low** → lost auctions, fewer impressions, less revenue
- **Too high** → expensive clicks, high spend per conversion, low ROAS

The optimizer finds the right setting for every keyword simultaneously — not in isolation.

### 1.2 Business Constraints — Priority Order

These constraints are not guidelines. They are the success criteria, ordered by severity:

**1. Budget — hard ceiling, non-negotiable**
Total projected daily spend per campaign must never exceed the campaign's daily budget. This is the dominant constraint. We will see in the EDA that every single campaign is currently far over budget — enforcing this is the primary job.

**2. ROAS — the north-star metric to maximize**
ROAS = Revenue / Spend. For every dollar spent, we want as many revenue dollars back as possible. There is no target ROAS to hit — higher is always better. Budget is the only ceiling.

**3. Bid stability — building advertiser trust**
Even a mathematically optimal large bid change can destroy advertiser confidence if it appears erratic. No keyword changes by more than ±50% per cycle.

**4. Explainability — every recommendation has a reason**
The `reason` column is a first-class output. A black box that produces numbers without reasoning won't be trusted in production.

**5. Graceful handling of sparse data**
Keywords with thin history must be treated carefully. They should not be silently given minimum bids — they deserve a principled estimate of their potential.

---
## 2. How Business Constraints Shape the Algorithm

The constraints above do not just influence parameters — they determine the entire architecture. Below I explain, from first principles, why each business constraint leads to a specific algorithm choice.

### 2.1 This Is a Portfolio Problem, Not a Pointwise Prediction Problem

The most important framing decision: **all keywords in a campaign share the same daily budget pool**. The bid for keyword A directly affects how much budget is left for keywords B, C, and D.

This rules out the naive approach of computing each keyword's "optimal bid" independently and then summing the projected spend — you inevitably land at 3× or 4× the budget and have to cut bids down anyway.

**The correct framing is a portfolio budget allocation problem:**

> Given a fixed daily budget per campaign, distribute it across keywords in proportion to each keyword's efficiency, so that the most profitable keywords receive more spend and the least profitable receive less.

This means:
- Bids are not calculated in isolation — they are derived from how much budget each keyword "deserves" relative to its campaign peers
- Budget constraint is satisfied by construction, before any individual bid is computed
- The relative ordering of keywords by efficiency is the core output; the bid is just the mechanical result of dividing allocated spend by expected clicks

### 2.2 No Train/Test Split — Full 30-Day Window

A chronological train/test split is correct for forecasting problems where we want to estimate how well the model generalises to future time periods. This is not a forecasting problem.

We are generating one operational recommendation per keyword for the *next* daily cycle. The right inputs are the best available performance estimates — which means using all 30 days of data, not discarding the most recent 9 days (the 30% most valuable signal) to create an artificial holdout.

**Using only days 1–21 would mean:**
- Bids for day 22 are based on stale data that ignores recent performance trends
- Keywords that improved or degraded in days 22–30 are treated as if that never happened

**Full 30-day window is used for all aggregations.** Model quality here is measured not by held-out prediction accuracy, but by whether the budget constraint is satisfied and whether projected ROAS improves.

### 2.3 Asymmetric Budget Allocation — Not Uniform Scaling

The naive budget fix is: if a campaign is spending 3× its budget, divide all bids by 3. This is a fundamental mistake because it penalises high-ROAS keywords identically to money-losing keywords.

**Example:** A campaign has 3 keywords:
- Keyword A: ROAS = 8× (cash-cow)
- Keyword B: ROAS = 1.5× (breakeven)
- Keyword C: ROAS = 0.2× (burning money)

Symmetric scaling cuts all three by the same factor. The optimal action is to concentrate budget on A, maintain B, and cut C deeply.

**The correct approach: score-weighted budget allocation**

Each keyword receives a share of the campaign's daily budget proportional to its efficiency score. High-score keywords get more; low-score keywords get less. The ranking is what matters — the exact bid follows mechanically from the budget share divided by expected daily clicks.

### 2.4 LLM Integration — Load-Bearing for Sparse Keywords

For keywords with abundant data (most of the portfolio), a purely behavioural efficiency score (ROAS, CVR, CTR) works well. But for keywords with thin or zero history, behavioural features are undefined — and a purely data-driven system would silently assign them zero budget forever.

**The solution: a hybrid efficiency score that blends behavioural evidence with an LLM semantic prior.**

For every keyword, Claude evaluates:
- `purchase_intent_score` — probability this query leads to a purchase on Amazon
- `competition_level` — typical CPC competition tier
- `estimated_efficiency` — overall ROAS potential for this keyword type, based on world knowledge

These feed into the hybrid score formula:

```
keyword_score = confidence × behavioral_score + (1 − confidence) × llm_efficiency
```

When `confidence = 0` (zero-history keyword), `keyword_score = llm_efficiency` — **the LLM is the sole allocation signal**.

This means "noise cancelling headphones" (high commercial intent) gets meaningfully more budget than "what is bluetooth" (informational) even with no historical data. The LLM is not decoration — remove it and every sparse keyword gets zero budget allocation regardless of its semantic potential.

### 2.5 Architecture Summary — What Business Logic Decided Before Touching the Data

| Design Decision | Reason |
|---|---|
| Full 30-day window, no train/test split | Operational optimisation — use best available signal, not artificially stale data |
| Portfolio allocation, not pointwise prediction | Keywords share a budget pool — isolation violates the core constraint |
| Asymmetric score-weighted budget allocation | Protects cash-cows while cutting money-losers — symmetric scaling destroys value |
| Hybrid behavioural + LLM score | LLM is the primary signal for sparse/cold-start keywords |
| ±50% stability cap | Business trust — erratic changes erode advertiser confidence |

The EDA below will validate these decisions and fill in the calibration details (confidence thresholds, ROAS distribution, how binding the budget constraint actually is).

---
## 3. Data Investigation — Questions & EDA

With the business context established, I now investigate the data to answer specific questions that will calibrate the algorithm. Each question below flows directly from a design decision or assumption in Section 2.

**Questions to investigate:**
1. What does missing data mean in this domain — random missingness or informative absence?
2. How sparse are conversions? → Determines how much we rely on own-data ROAS vs. LLM prior
3. Does `current_bid` actually vary over time? → Validates or refutes the bid-response curve approach
4. What does the ROAS distribution look like? → Calibrates the ±50% change cap
5. Are categorical features balanced? → Affects encoding decisions
6. Are campaigns currently over or under budget? → Determines how binding the budget constraint is
7. Is there a temporal trend in ROAS? → Determines whether flat aggregation is valid

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

BID_MIN = 0.20
BID_MAX = 15.00

In [ ]:
df = pd.read_csv('campaign_data.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"Dataset shape:  {df.shape}")
print(f"Date range:     {df['date'].min().date()} to {df['date'].max().date()} ({df['date'].nunique()} days)")
print(f"Keywords:       {df['keyword_id'].nunique()}")
print(f"Campaigns:      {df['campaign_id'].nunique()}")
print(f"Match types:    {df['match_type'].value_counts().to_dict()}")
print()
print("Column types:")
print(df.dtypes)
print()
print("First 3 rows:")
df.head(3)

### Q1 — Missing Data: Random or Informative?

In advertising data, absence is rarely random. There are two distinct kinds of "zero" that mean completely different things:
- A row with `revenue = 0, clicks = 0` → keyword **ran** but did not get any clicks. Valid negative signal — keep it.
- A completely absent day for a keyword → keyword **did not run** (budget exhausted, bid lost the auction). This is informative absence — do not fill with zero.

If we treat absent days as zero-revenue days, we systematically underestimate `avg_daily_spend` and `avg_daily_clicks`, which will break the budget enforcement step later.

In [ ]:
# Q1: Structural missing values and day coverage per keyword
print("=== Structural Missing Values ===")
print(df.isnull().sum())
print()

days_per_kw = df.groupby('keyword_id')['date'].nunique()
print("=== Day Coverage per Keyword (out of 30 days) ===")
print(f"All 30 days active:  {(days_per_kw == 30).sum()} keywords")
print(f"20-29 days active:   {((days_per_kw >= 20) & (days_per_kw < 30)).sum()} keywords")
print(f"< 7 days active:     {(days_per_kw < 7).sum()} keywords")
print()

print("=== Zero-Value Rows (valid signals, not missing) ===")
for col in ['revenue', 'clicks', 'conversions']:
    n = (df[col] == 0).sum()
    print(f"  {col} = 0:  {n} rows ({n/len(df)*100:.1f}%)")

**Finding — Missing data:**
No structural missing values — all columns fully populated. 186 of 200 keywords appear all 30 days; 2 keywords appear only ~3 days (new or paused keywords). 54% of rows have zero revenue — this is valid signal (keyword ran, nobody converted), not missing data.

**Algorithm implication:** No imputation needed. Aggregate over `days_active` (not calendar days) to get accurate per-day rates. The 2 sparse keywords will automatically receive near-zero confidence scores.

### Q2 — Conversion Sparsity: How Much Own-Data ROAS Signal Do We Have?

This is the most critical question for calibrating the behavioural vs. LLM prior blend. If most keywords have very few conversions, ROAS estimates are too noisy to trust and we must rely heavily on the LLM prior. If conversions are abundant, own-data signals dominate.

In [ ]:
# Q2: Conversions and clicks per keyword across full 30-day window
kw_agg = df.groupby(['keyword_id','keyword_text','campaign_id','campaign_name','match_type']).agg(
    total_spend           = ('spend',                'sum'),
    total_revenue         = ('revenue',              'sum'),
    total_clicks          = ('clicks',               'sum'),
    total_impressions     = ('impressions',          'sum'),
    total_conversions     = ('conversions',          'sum'),
    days_active           = ('date',                 'nunique'),
    current_avg_bid       = ('current_bid',          'mean'),
    campaign_daily_budget = ('campaign_daily_budget','last'),
).reset_index()

print("=== Conversions per Keyword (full 30-day window) ===")
buckets = [(0,0), (1,4), (5,9), (10,None)]
for lo, hi in buckets:
    if hi is None:
        mask = kw_agg['total_conversions'] >= lo
        label = f">= {lo}"
    elif lo == hi:
        mask = kw_agg['total_conversions'] == lo
        label = f"= {lo}"
    else:
        mask = (kw_agg['total_conversions'] >= lo) & (kw_agg['total_conversions'] <= hi)
        label = f"{lo}-{hi}"
    n = mask.sum()
    print(f"  Conversions {label}:  {n} keywords ({n/len(kw_agg)*100:.0f}%)")

print()
print("=== Clicks per Keyword ===")
print(f"  < 30 clicks:   {(kw_agg['total_clicks'] < 30).sum()} keywords")
print(f"  >= 100 clicks: {(kw_agg['total_clicks'] >= 100).sum()} keywords")
print(f"  Range: {kw_agg['total_clicks'].min()} – {kw_agg['total_clicks'].max()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
kw_agg['total_conversions'].hist(bins=30, ax=axes[0])
axes[0].set_title('Conversions per Keyword (30 days)')
axes[0].set_xlabel('Total Conversions')
axes[0].set_ylabel('# Keywords')

kw_agg['total_clicks'].hist(bins=40, ax=axes[1])
axes[1].axvline(30, color='red', linestyle='--', label='30-click threshold')
axes[1].set_title('Clicks per Keyword (30 days)')
axes[1].set_xlabel('Total Clicks')
axes[1].legend()
plt.tight_layout()
plt.show()

**Finding — Conversion sparsity:**
87% of keywords have ≥10 conversions over 30 days. Only 3 keywords (2%) have zero conversions. 99% have ≥100 clicks. This is a data-rich portfolio — much more signal than expected.

**Algorithm implication:** Own-data ROAS is reliable for the vast majority of keywords. The LLM prior is a fallback for the 3% edge cases (zero conversions), not the main mechanism. Confidence thresholds of 30 clicks and 7 active days will produce confidence ≈ 1 for nearly the entire portfolio — which is correct, the data is genuinely trustworthy.

**Also rules out DNN:** 200 keyword-level observations is far too small for neural networks on tabular data. Score-based allocation is the right fit.

### Q3 — Bid Variability: Does `current_bid` Change Over Time?

This question determines whether we can learn a per-keyword bid-response curve (how ROAS changes as the bid changes). If bids are static, there is zero within-keyword variation and curve fitting is impossible. If bids vary, there might be some signal — though we expect 3–5 distinct values is still too sparse.

In [ ]:
# Q3: Bid variability per keyword
bid_stats = df.groupby('keyword_id')['current_bid'].agg(['nunique','std','min','max'])

print("=== Bid Variability per Keyword ===")
print(f"  Static bid (1 unique value):  {(bid_stats['nunique']==1).sum()} keywords ({(bid_stats['nunique']==1).mean()*100:.0f}%)")
print(f"  2 unique values:              {(bid_stats['nunique']==2).sum()} keywords")
print(f"  3+ unique values:             {(bid_stats['nunique']>=3).sum()} keywords ({(bid_stats['nunique']>=3).mean()*100:.0f}%)")
print(f"  Max unique bids (one kw):     {bid_stats['nunique'].max()}")
print()
print(bid_stats['nunique'].describe().rename('unique_bid_values').to_frame().T.round(1))

**Finding — Bid variability:**
96% of keywords have 3+ unique bid values — bids are actively adjusted by the existing system. Only 7 keywords (4%) have a completely static bid.

**Algorithm implication:** Even with variation, 3–5 distinct bid values over 30 days is insufficient to learn a reliable bid-response curve per keyword. The range of variation is too narrow and too noisy from day-to-day effects (day-of-week, competitor activity, seasonality). Score-weighted budget allocation is the right approach — we derive the bid from the budget allocation, not from curve-fitting.

### Q4 — ROAS Distribution: What Range Are We Operating In?

The ROAS distribution tells us two things: (1) how wide the spread is, which determines how aggressive we can be with bid changes, and (2) whether most keywords are currently profitable or money-losing.

In [ ]:
# Q4: ROAS distribution across keywords
kw_agg['roas'] = np.where(kw_agg['total_spend'] > 0,
                           kw_agg['total_revenue'] / kw_agg['total_spend'], np.nan)

roas_valid = kw_agg['roas'].dropna()
print("=== ROAS Distribution (keyword-level, 30-day aggregate) ===")
print(f"  Keywords with no spend (undefined ROAS): {kw_agg['roas'].isna().sum()}")
print(f"  ROAS = 0 (spend but no revenue):         {(kw_agg['roas']==0).sum()}")
print(f"  ROAS > 0:                                {(kw_agg['roas']>0).sum()}")
print()
print(f"  Range:   {roas_valid.min():.2f} — {roas_valid.max():.2f}")
print(f"  Mean:    {roas_valid.mean():.2f}")
print(f"  Median:  {roas_valid.median():.2f}")
print(f"  Std:     {roas_valid.std():.2f}   (std > mean — very high variance)")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
roas_valid.clip(0, 10).hist(bins=40, ax=axes[0])
axes[0].axvline(1.0, color='red', linestyle='--', label='Break-even (ROAS=1)')
axes[0].set_title('ROAS Distribution (clipped at 10×)')
axes[0].set_xlabel('ROAS')
axes[0].set_ylabel('# Keywords')
axes[0].legend()

axes[1].scatter(kw_agg['total_clicks'], kw_agg['roas'].clip(0,10), alpha=0.4, s=20)
axes[1].axhline(1.0, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('Total Clicks')
axes[1].set_ylabel('ROAS (clipped at 10×)')
axes[1].set_title('ROAS vs. Click Volume')
plt.tight_layout()
plt.show()

**Finding — ROAS distribution:**
ROAS ranges from 0.1 to 9.7 with mean 1.3 and std 1.5 — standard deviation exceeds the mean, indicating extremely high variance. Median is below 1, meaning most keywords are currently losing money or barely breaking even. Only a minority of keywords are genuinely profitable.

**Algorithm implication:** The high variance confirms the need for the ±50% stability cap — raw ROAS ratios would suggest 5×–10× bid changes that would destabilise campaigns. The wide spread also means score-weighted allocation will produce meaningful differentiation: top-ROAS keywords will receive dramatically more budget than bottom-ROAS ones.

### Q5 — Categorical Balance: Are Features Skewed?

We check whether match types and campaign sizes are balanced before deciding on encoding strategies.

In [ ]:
# Q5: Match type and campaign size balance
print("=== Match Type Distribution ===")
print(df['match_type'].value_counts())
print()

kw_per_camp = df.groupby('campaign_name')['keyword_id'].nunique().sort_values()
print("=== Keywords per Campaign ===")
print(kw_per_camp)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['match_type'].value_counts().plot(kind='bar', ax=axes[0])
axes[0].set_title('Match Type Distribution')
axes[0].set_ylabel('# Rows')

kw_per_camp.plot(kind='barh', ax=axes[1])
axes[1].set_title('Keywords per Campaign')
axes[1].set_xlabel('# Keywords')
plt.tight_layout()
plt.show()

**Finding — Categorical balance:**
Match types are well balanced: broad 28%, phrase 36%, exact 35%. Keywords per campaign range from 11 to 27 — no extreme imbalance.

**Algorithm implication:** No grouping or special handling needed. All match types can be encoded directly as features. Campaign sizes are similar enough that the score-weighted allocation is fair across campaigns.

### Q6 — Budget Utilisation: Are Campaigns Over or Under Budget?

This is the most operationally critical finding. The answer determines how much the budget enforcement step will impact the final recommendations.

In [ ]:
# Q6: Current spend vs daily budget per campaign
daily_camp = df.groupby(['campaign_id','campaign_name','date']).agg(
    daily_spend  = ('spend',                'sum'),
    daily_budget = ('campaign_daily_budget','last'),
).reset_index()

camp_summary = (
    daily_camp.groupby(['campaign_name','daily_budget'])
    .agg(avg_daily_spend=('daily_spend','mean'), max_daily_spend=('daily_spend','max'))
    .reset_index()
)
camp_summary['utilisation'] = camp_summary['avg_daily_spend'] / camp_summary['daily_budget']
camp_summary = camp_summary.sort_values('utilisation', ascending=False)

print("=== Campaign Budget Utilisation ===")
print(camp_summary[['campaign_name','daily_budget','avg_daily_spend','utilisation']].to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 4))
camp_summary.set_index('campaign_name')['utilisation'].sort_values().plot(kind='barh', ax=ax)
ax.axvline(1.0, color='red', linestyle='--', linewidth=2, label='Budget ceiling')
ax.set_xlabel('Avg Daily Spend / Daily Budget')
ax.set_title('Campaign Budget Utilisation — All campaigns are over budget')
ax.legend()
plt.tight_layout()
plt.show()

**Finding — Budget utilisation (most critical finding):**
Every single campaign is over its daily budget. The lowest overspend is 1.6× (Beauty & Personal Care); the highest is 4.8× (Pet Supplies, spending ~$572/day against a $120 budget).

**Algorithm implication:** Budget enforcement is not an edge case — it is the dominant output force. After computing efficiency scores and deriving bids, we must bring every campaign into compliance. This also means the `avg_daily_clicks × recommended_bid` projected spend estimate must be accurate: errors here directly translate to budget violations. We must aggregate clicks over active days only (per Q1), not calendar days.

### Q7 — Temporal Trend: Is ROAS Stable Over the 30-Day Window?

If ROAS trends significantly upward or downward over the window, recent data is more representative than old data and we should apply recency weighting. If performance is stable, flat aggregation over all 30 days is valid.

In [ ]:
# Q7: Daily portfolio ROAS trend
daily_all = df.groupby('date').agg(
    total_spend   = ('spend',   'sum'),
    total_revenue = ('revenue', 'sum'),
).reset_index()
daily_all['roas'] = daily_all['total_revenue'] / daily_all['total_spend'].replace(0, np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(daily_all['date'], daily_all['roas'])
axes[0].set_title('Daily Portfolio ROAS — 30-Day Window')
axes[0].set_ylabel('ROAS')
axes[0].tick_params(axis='x', rotation=45)

axes[1].plot(daily_all['date'], daily_all['total_spend'], label='Spend')
axes[1].plot(daily_all['date'], daily_all['total_revenue'], label='Revenue')
axes[1].set_title('Daily Spend vs Revenue')
axes[1].legend()
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

corr, pval = stats.spearmanr(range(len(daily_all)), daily_all['roas'].fillna(0))
print(f"ROAS temporal trend — Spearman r = {corr:.3f}, p-value = {pval:.3f}")
print(f"Daily ROAS range: {daily_all['roas'].min():.2f} – {daily_all['roas'].max():.2f}, "
      f"std = {daily_all['roas'].std():.2f}")
conclusion = "Significant trend detected — consider recency weighting" if pval < 0.05 else              "No significant trend — flat 30-day aggregation is valid"
print(f"Conclusion: {conclusion}")

**Finding — Temporal stability:**
Spearman r ≈ 0.17, p ≈ 0.36 — no statistically significant trend in ROAS over the 30-day window. Daily fluctuation is noise, not signal.

**Algorithm implication:** Flat aggregation across all 30 days is valid. No recency weighting needed. This simplifies the feature engineering and avoids overfitting to a short recent window that might be atypical.

---
## 4. EDA Findings & Algorithm Decisions

Each EDA finding either validates a prior assumption or forces a concrete design decision. This section makes the mapping explicit — this is where the data shapes the algorithm.

| Finding | Key Result | Algorithm Decision |
|---|---|---|
| **Q1 — Missing data** | No structural NaNs. 54% of rows have zero revenue (valid signal). 2 keywords active only ~3 days. | Aggregate over `days_active` only — never calendar days. Keep zero-revenue rows. 2 sparse keywords get confidence ≈ 0. |
| **Q2 — Conversion sparsity** | 87% have ≥10 conversions; only 3 keywords (2%) have zero. 99% have ≥100 clicks. | Own-data ROAS is reliable for most keywords. LLM prior is a fallback, not primary mechanism. Confidence thresholds (30 clicks, 7 days) are correctly calibrated. |
| **Q3 — Bid variability** | 96% have 3+ unique bid values. Bids are actively adjusted. | 3–5 distinct bid levels per keyword is too sparse for per-keyword bid-response curve fitting. Confirms portfolio allocation approach. |
| **Q4 — ROAS distribution** | Range 0.1–9.7, std > mean. Median < 1. Extremely high variance. | ±50% change cap is essential — raw ratios would suggest extreme bid changes. Score-weighted allocation will create meaningful differentiation across keywords. |
| **Q5 — Categorical balance** | Match types balanced. Campaign sizes 11–27 keywords. | No special encoding or grouping needed. |
| **Q6 — Budget utilisation** | **All 10 campaigns 1.6×–4.8× over daily budget.** | Budget enforcement is the dominant output force. `avg_daily_clicks × bid` projected spend must be accurate. Aggregate clicks over active days only. |
| **Q7 — Temporal trend** | Spearman r = 0.17, p = 0.36. No significant trend. | Flat 30-day aggregation is valid. No recency weighting needed. |
| **Data scale** | 200 keyword-level observations after aggregation. | Rules out DNN and gradient boosting. Score-based allocation is the right fit at this scale. |

### Confidence Score Design — Calibrated by Q1, Q2, Q4

The confidence score controls how much we trust a keyword's own behavioural statistics vs. the LLM prior:

```
confidence = sqrt( min(total_clicks / 30, 1)  ×  min(days_active / 7, 1) )
```

- **Geometric mean** (sqrt of product): both click volume *and* temporal stability must hold. A keyword with 500 clicks on a single day has near-zero temporal stability — geometric mean captures this correctly.
- **Denominator is 30 clicks / 7 days** (fixed business thresholds, not derived from the dataset): using dataset statistics as thresholds would cause data leakage if we later evaluate hold-out performance.
- **Q2 calibration:** 99% of keywords have ≥100 clicks and most appear all 30 days. Confidence ≈ 1 for nearly everyone — which is correct, the data is genuinely reliable for most of the portfolio.
- **Q1 calibration:** The 2 keywords active only 3 days will have confidence ≈ 0.65 (3/7 day factor). They receive a blended score that still draws on their own data but leans more on the LLM prior.
- **Q2 + Q6 calibration:** For zero-conversion keywords, `behavioral_score` will be low (cvr = 0, roas = 0) regardless of confidence. The LLM prior provides a floor that prevents these keywords from being completely starved of budget.

### Next Steps

The EDA has validated all architectural decisions from Section 2 and calibrated the key parameters:
- Full 30-day window confirmed (Q7)
- Score-weighted portfolio allocation confirmed (Q3, Q6)
- LLM prior necessity confirmed for sparse keywords (Q2)
- ±50% cap necessity confirmed (Q4)
- Confidence threshold calibration confirmed (Q2)
- Projected spend formula confirmed (Q1, Q6)

**Sections 3–5 (to be built with user approval):**
- Section 3: Feature engineering (aggregate kw_train, compute confidence, derive behavioral score)
- Section 4: LLM semantic enrichment + hybrid score
- Section 5: Portfolio budget allocation algorithm + output generation

---
## 5. Model Architecture & Feature Engineering Plan

This section translates every EDA finding into a concrete engineering decision. Each stage, each feature, and each formula is grounded in a specific observation from Section 3.

```
[ Stage 1: Behavioral Feature Engineering    ]
                |
                v
[ Stage 2: Campaign Contextualization         ]
                |
                v
[ Stage 3: LLM Semantic Feature Layer         ]
                |
                v
[ Stage 4: Unified Keyword Score              ]
                |
                v
[ Stage 5: Asymmetric Portfolio Optimization  ]
                |
                v
[ Stage 6: Marketplace Compliance & Output    ]
```

### Why a Score-Based Heuristic and Not a Classic ML Model

This is a deliberate choice driven by three hard constraints the EDA exposed.

---

**Constraint 1 — No target variable (Q3)**

Supervised learning requires a label: the "correct" bid for each keyword. We never observe that. We only observe revenue at the bids that were *actually* used. The counterfactual — "what would revenue have been at bid $2.50 instead of $1.80?" — is unobserved. Without counterfactual outcomes, there is no target to train against.

---

**Constraint 2 — No bid-response curve (Q3)**

The natural proxy target is a per-keyword bid elasticity curve: how does ROAS change as bid changes? This requires bid variation. Q3 found 3–5 unique bid values per keyword over 30 days — far too sparse to fit a reliable curve. Fitting a regression on 3–5 points with day-to-day noise (competitor activity, seasonality) would produce confident but meaningless coefficients.

---

**Constraint 3 — Sample size rules out tabular ML (Q2)**

After aggregation, ~200 keyword-level observations. Gradient boosting and neural networks on tabular data require thousands of samples to generalise. At 200 rows, any model complex enough to capture non-linear ROAS interactions would overfit immediately. Ridge regression is technically feasible but would simply re-learn what the score formula already encodes — no new information.

---

**What the EDA pointed to instead**

The correct framing is a **portfolio allocation problem, not a prediction problem**. The question is not "predict the optimal bid" — it is "given a fixed budget, which keywords deserve more of it?" That question is answered by *ranking*, not regression. A well-calibrated score — grounded in ROAS, CVR, and semantic signal — gives the ranking. The bid follows mechanically from the allocated budget share.

The LLM components are where AI adds genuine value that pure heuristics cannot: reading raw keyword text to estimate commercial potential for keywords with no behavioural history. That is the load-bearing AI contribution.

### Stage 1 — Behavioral Feature Engineering

**Source:** EDA Q1, Q2, Q4, Q5, Q7

The raw dataset is a 30-day daily log (~6,000 rows). The first engineering step collapses this into one row per keyword — a flat profile matrix that becomes the input to every downstream stage.

**Critical aggregation rule from Q1:** divide by `days_active = CountUnique(date)`, not by 30 calendar days. A keyword that ran 10 days and spent $50 total has `avg_daily_spend = $5.00`. Dividing by 30 gives $1.67 — a 3x underestimate that flows directly into the budget enforcement step and causes budget violations.

| Feature | Formula | EDA Justification |
|---|---|---|
| `historical_roas` | total_revenue / total_spend | Q2: 87% of keywords have >=10 conversions — ROAS is reliable for most of the portfolio |
| `cvr` | total_conversions / total_clicks | Q2: captures buyer quality independently of bid level; stable signal |
| `ctr` | total_clicks / total_impressions | Ad relevance proxy — lower weight in scoring; partially bid-driven (higher bid = better placement = inflated CTR) |
| `avg_daily_clicks` | total_clicks / days_active | Q1: informative absence rule; Q6: feeds projected_spend = avg_daily_clicks x bid in Stage 5 |
| `avg_daily_spend` | total_spend / days_active | Q1: same informative absence rule |
| `current_avg_bid` | mean(current_bid) over 30 days | Q7: no temporal trend — 30-day mean is stable; serves as anchor for the +-50% stability cap |
| `match_type` | Ordinal: exact=1.0, phrase=0.8, broad=0.5 | Q5: balanced distribution; ordering reflects specificity and conversion intent |
| `confidence` | sqrt( min(clicks/30, 1) x min(days_active/7, 1) ) | Q1, Q2: geometric mean of click saturation and temporal stability |

**Why geometric mean for confidence?**
A keyword with 500 clicks on a single promotional day has click saturation but zero temporal stability. Arithmetic mean gives it confidence=0.5. Geometric mean gives near-zero — which is correct, because one day of data is not enough to trust. Both dimensions must hold simultaneously.

### Stage 2 — Campaign Contextualization

**Source:** EDA Q4 (ROAS range 0.1–9.7, std > mean)

Q4 showed ROAS is extremely volatile — std exceeds the mean. Ranking keywords by absolute ROAS against a fixed threshold is meaningless: a keyword at ROAS=2.0 is a top performer in one campaign and a laggard in another.

The fix: compute **campaign-level spend-weighted ROAS** and express each keyword's performance relative to its own budget pool:

```
campaign_avg_roas  =  sum(campaign revenue) / sum(campaign spend)
roas_vs_campaign   =  keyword historical_roas / campaign_avg_roas
```

- `roas_vs_campaign > 1` → keyword outperforms its campaign peers → deserves more budget
- `roas_vs_campaign < 1` → keyword underperforms → candidate for reduction

This is what makes the scoring asymmetric: it ranks keywords within the same competitive context, not against a global absolute benchmark.

### Stage 3 — LLM Semantic Feature Layer

**Source:** Business constraint 5 (sparse data); Q2 (3 zero-conversion keywords); architectural decision 2.4

Standard tabular features are structurally blind to commercial context. A tabular model cannot distinguish a high-margin luxury query from a low-margin commodity query — both look identical until enough conversion data accumulates. The LLM layer solves this by reading the raw `keyword_text` directly.

This layer has two components:

---

**Component A — Embedding Layer (fully offline, no API key)**

Model: `all-MiniLM-L6-v2` via `sentence-transformers`. Runs locally, zero external dependencies.

Each `keyword_text` is encoded into a 384-dimensional dense semantic vector. These embeddings power two distinct features:

**A1 — Specificity Score (centroid distance)**

Compute the centroid (mean vector) of all keyword embeddings in the dataset. Score each keyword by its cosine distance from that centroid:
- Far from centroid = semantically specific and unique = long-tail, high-converting
- Close to centroid = generic head term = high competition, lower conversion efficiency

**Why centroid distance for Amazon specifically?** On Amazon, commercial intent is implicit in every search — nobody opens Amazon to research what headphones are. The signal that differentiates keyword value is *specificity*, not intent. "Sony WH-1000XM5 noise cancelling over-ear" and "headphones" are both transactional, but the first converts at 3–5× the rate because the shopper has already decided. Centroid distance captures this with no hardcoded anchors.

---

**A2 — Neighbor ROAS (KNN in embedding space)**

For sparse keywords (low confidence), instead of relying solely on LLM priors, we directly borrow the historical ROAS signal from semantically similar keywords that have rich data.

Algorithm:
1. For each keyword, find its k=5 nearest neighbors by cosine similarity in embedding space
2. Compute similarity-weighted average ROAS from those neighbors:

```
neighbor_roas = sum(similarity_i × historical_roas_i) / sum(similarity_i)
```

3. This becomes the behavioral prior for sparse keywords — grounded in actual historical performance of similar keywords, not just a semantic guess

**Why this matters:** A keyword with 3 days of history doesn't get a pure LLM estimate — it gets the actual earned ROAS of the 5 most semantically similar keywords in the portfolio, weighted by how similar they are. If those neighbors converted at ROAS 4.2, the sparse keyword's prior reflects that directly.

This is strictly stronger than centroid distance alone for cold-start: specificity tells you *how long-tail* the keyword is; neighbor ROAS tells you *what similar keywords actually earned*.

---

**Component B — Claude Structured Metadata (API, cached to JSON)**

Model: `claude-haiku-4-5-20251001`. Called once per unique keyword, cached to `keyword_metadata_cache.json`. Every subsequent run loads from cache — zero latency, fully reproducible.

**The cache is committed to the repository.** Reviewers reproduce results without an API key. Keywords absent from cache fall back to Component A only — graceful degradation, not a hard failure.

For each keyword, Claude returns:

```json
{
  "margin_tier":           "premium | mid-range | commodity",
  "funnel_stage":          "awareness | consideration | decision",
  "competition_intensity": "low | medium | high"
}
```

**Why these three fields?**

- **`margin_tier`:** The optimal bid ceiling is a direct function of margin — a bid profitable for a premium product is a money-loser for a commodity. Tabular ROAS history cannot separate margin from volume. Claude can.
- **`funnel_stage`:** Decision-stage keywords (specific product + attributes) convert immediately; awareness-stage keywords require multiple exposures. Budget should concentrate on decision-stage keywords where the conversion is imminent.
- **`competition_intensity`:** Determines the bid *floor* needed to win auctions at all. Feeds the bid floor modifier in Stage 5, not the keyword score.

These fields are ordinally encoded and merged into the keyword feature matrix alongside Stage 1 behavioral features.

### Stage 4 — Unified Keyword Score

**Source:** All EDA findings converging; architectural decisions 2.3, 2.4

The score is a single continuous number summarising each keyword's expected return on investment. It is the sole input to Stage 5 — budget allocation is based entirely on this rank ordering.

**Behavioral component — weighted additive sum:**

```
behavioral_score = 0.60 × roas_vs_campaign_norm
                 + 0.25 × cvr_norm
                 + 0.15 × match_type_factor
```

Each term is **percentile-rank normalised** within the full dataset before blending, placing all factors on [0, 1]. Weights reflect the optimization target: ROAS dominates (0.60) because it is the metric being optimised; CVR (0.25) captures buyer quality independent of bid level; match type (0.15) acts as a prior on specificity.

**Why not CTR?** CTR is partially bid-driven — a higher bid wins better ad placement, which mechanically inflates CTR regardless of keyword quality. Including CTR in the score would reward keywords for spending more, creating a circular signal. Dropped from scoring; retained in the feature table as a diagnostic column only.

**Why additive, not multiplicative?** Multiplication collapses to zero if any single term is weak — a keyword with zero conversions and good ROAS would score zero regardless of potential. The weighted sum ensures each signal contributes independently.

**Hybrid score formula:**

```
keyword_score = confidence × behavioral_score
              + (1 − confidence) × (0.60 × neighbor_roas_norm + 0.40 × llm_specificity)
              + llm_margin_modifier
              + llm_funnel_modifier
```

When `confidence` is low, the score falls back to a blend of two semantic signals: `neighbor_roas_norm` (actual earned ROAS of the 5 most similar keywords, 0.60 weight) and `llm_specificity` (how long-tail the keyword is, 0.40 weight). The KNN lookup is what makes the embedding layer genuinely load-bearing for sparse keywords.

**The confidence weight is the load-bearing mechanism:**

| confidence value | interpretation | score driven by |
|---|---|---|
| 1.0 | data-rich, 30+ clicks, 7+ active days | 100% own behavioral ROAS, CVR, match type |
| 0.5 | moderately observed | 50% own data + 50% neighbor ROAS + specificity blend |
| 0.0 | zero history (cold-start / 3-day keyword) | 100% neighbor ROAS + LLM specificity — no historical noise |

**LLM modifier values:**

| Field | Value | Modifier |
|---|---|---|
| `margin_tier` | commodity | +0.00 |
| `margin_tier` | mid_range | +0.05 |
| `margin_tier` | premium | +0.10 |
| `funnel_stage` | awareness | +0.00 |
| `funnel_stage` | consideration | +0.05 |
| `funnel_stage` | decision | +0.10 |

`competition_intensity` is **not** a score additive — it feeds the bid floor modifier in Stage 5.

### Stage 5 — Asymmetric Portfolio Optimization

**Source:** EDA Q6 (all 10 campaigns 1.6×–4.8× over budget); architectural decision 2.3

Q6 is the most operationally critical finding. Every campaign is massively over its daily budget. Budget enforcement will determine the final recommendations more than any other stage.

---

**Step 1 — Score-to-bid derivation (budget satisfied by construction):**

```
expected_clicks = max(avg_daily_clicks, 0.5)   # floor prevents division by zero
raw_bid = (keyword_score / sum(campaign_keyword_scores)) × campaign_daily_budget / expected_clicks
```

`avg_daily_clicks` can be zero for keywords with impressions but no clicks. Using a floor of 0.5 expected clicks/day instead of zero avoids division-by-zero while keeping the allocated budget share meaningful. These keywords will receive a low raw bid (their score is typically low too) and are flagged in the reason column.

Each keyword receives a share of the campaign's daily budget proportional to its efficiency score, then divided by expected daily clicks to convert allocated spend into a per-click bid. The budget constraint is satisfied by construction — the slash loop below is a safety net for rounding errors and click-rate variance, not the primary budget mechanism.

---

**Step 2 — Stability cap:**

```
raw_bid = clip(raw_bid, current_avg_bid × 0.50, current_avg_bid × 1.50)
```

Applied after bid derivation — not before, when there is no bid yet. This enforces the ±50% movement constraint and prevents the portfolio optimization from moving bids too aggressively even when the score justifies it. Capping here (before the floor modifier) means the floor in Step 3 can still override the cap from below if the cap would produce a sub-floor bid.

---

**Step 3 — Competition bid floor modifier:**

```
floor_bid = max(0.20, 0.20 × (1 + 0.15 × competition_ordinal))
raw_bid   = max(raw_bid, floor_bid)
```

where `competition_ordinal`: low = 0, medium = 1, high = 2.
Floor values: low → $0.20, medium → $0.23, high → $0.26.

`competition_intensity` is not a score component — it sets the minimum bid needed to enter the auction, not the relative ranking of keywords.

---

**Step 4 — Asymmetric safety slash (if campaign still over budget after rounding):**

1. Compute `projected_spend = raw_bid × expected_clicks` per keyword
2. If `sum(projected_spend) ≤ campaign_daily_budget` — no action needed
3. If over budget — asymmetric slash:
   - Sort keywords by `keyword_score` ascending (worst performers first)
   - Drive the lowest-score keyword's bid down toward its `floor_bid`
   - **`floor_bid` enforced inside the loop** — never go below on any iteration
   - Recompute projected spend after each reduction using the floored bid
   - Stop as soon as the campaign is within budget
   - **High-score keywords are never touched**

**Why the floor must be enforced inside the loop, not after:**
If bids were allowed to go below `floor_bid` during optimization and then clipped back up at a later stage, those upward corrections would increase projected spend and re-violate the budget constraint the loop just satisfied.

**Edge case — budget floor exceeded:**
If every keyword in a campaign reaches its `floor_bid` and projected spend still exceeds the daily budget, the constraint cannot be satisfied. In this case: leave all keywords at `floor_bid` and flag the campaign in the reason column with *"budget floor reached — campaign cannot be brought within budget at marketplace minimum bids."*

**Safety net:**
If the slash loop completes but projected spend is still marginally over budget due to the floor constraint, apply proportional scale-down to the whole campaign as a last resort, again enforcing `floor_bid` per keyword after scaling.

### Stage 6 — Marketplace Compliance & Output

1. **Ceiling clip only:** `recommended_bid = min(bid, 15.00)` — the $0.20 floor is already guaranteed by Stage 5's optimization loop. The only remaining compliance risk is bids above $15.00, which the stability cap (`current_avg_bid x 1.5`) can produce for high-bid keywords.
2. **Reason column:** one sentence per keyword documenting the key signals (ROAS vs campaign, confidence level, LLM metadata used) and the budget action taken. Keywords that hit the $0.20 floor are explicitly flagged.
3. **Output:** `keyword_id`, `keyword_text`, `current_avg_bid`, `recommended_bid`, `reason_or_score` written to `bid_recommendations.csv`

### Complete Feature Engineering Plan

| Feature | Stage | Type | Formula / Source | EDA Anchor |
|---|---|---|---|---|
| `historical_roas` | 1 | Behavioral | total_revenue / total_spend | Q2: reliable conversions |
| `cvr` | 1 | Behavioral | total_conversions / total_clicks | Q2: buyer quality signal |
| `ctr` | 1 | Diagnostic | total_clicks / total_impressions | Diagnostic only — bid-driven, excluded from score |
| `avg_daily_clicks` | 1 | Behavioral | total_clicks / days_active | Q1: informative absence |
| `avg_daily_spend` | 1 | Behavioral | total_spend / days_active | Q1: informative absence |
| `current_avg_bid` | 1 | Anchor | mean(current_bid) | Stability cap anchor |
| `match_type_factor` | 1 | Ordinal | exact=1.0, phrase=0.8, broad=0.5 | Q5: balanced, ordered |
| `confidence` | 1 | Weight | sqrt(clicks/30 × days/7), capped at 1 | Q1, Q2 calibration |
| `campaign_avg_roas` | 2 | Context | spend-weighted ROAS per campaign | Q4: normalises variance |
| `roas_vs_campaign` | 2 | Relative | historical_roas / campaign_avg_roas | Q4: within-pool ranking |
| `llm_specificity` | 3A | Semantic | cosine distance from embedding centroid | Long-tail vs generic |
| `neighbor_roas` | 3A | Semantic | similarity-weighted ROAS of k=5 nearest neighbors in embedding space | Cold-start prior from real historical data |
| `llm_margin_tier` | 3B | Semantic | Claude JSON → commodity=+0.00, mid_range=+0.05, premium=+0.10 | Margin blind spot in tabular data |
| `llm_funnel_stage` | 3B | Semantic | Claude JSON → awareness=+0.00, consideration=+0.05, decision=+0.10 | Conversion proximity signal |
| `llm_competition` | 3B | Floor mod | Claude JSON → floor: low=$0.20, medium=$0.23, high=$0.26 | Bid floor only — not a score component |

> Code implementation: **Section 6** (behavioral engineering) and **Section 7** (LLM layer) in [bid_optimizer.ipynb](bid_optimizer.ipynb)

---
## 6. Feature Engineering — Stages 1 & 2

We now collapse the 30-day daily log into one row per keyword. Every rate divides by `days_active`
(not by 30 calendar days) — the informative-absence rule from Q1: a keyword absent on a day
did not run, so that day carries no signal.

> Architecture plan: **Section 5, Stage 1 & 2** in this notebook

In [ ]:
import pandas as pd
import numpy as np

BID_MIN = 0.20
BID_MAX = 15.00

df = pd.read_csv('campaign_data.csv')
df['date'] = pd.to_datetime(df['date'])

# ── Stage 1: aggregate to keyword-level profile ───────────────────────────────
kw = df.groupby(
    ['keyword_id', 'keyword_text', 'campaign_id', 'campaign_name', 'match_type']
).agg(
    total_spend           = ('spend',                'sum'),
    total_revenue         = ('revenue',              'sum'),
    total_clicks          = ('clicks',               'sum'),
    total_impressions     = ('impressions',          'sum'),
    total_conversions     = ('conversions',          'sum'),
    days_active           = ('date',                 'nunique'),
    current_avg_bid       = ('current_bid',          'mean'),
    campaign_daily_budget = ('campaign_daily_budget','last'),
).reset_index()

# Rates divided by days_active (Q1: informative absence)
kw['avg_daily_clicks'] = kw['total_clicks'] / kw['days_active']
kw['avg_daily_spend']  = kw['total_spend']  / kw['days_active']

# Performance features (zero-safe)
kw['historical_roas'] = np.where(kw['total_spend']       > 0,
                                  kw['total_revenue']     / kw['total_spend'],       0.0)
kw['cvr']             = np.where(kw['total_clicks']       > 0,
                                  kw['total_conversions'] / kw['total_clicks'],      0.0)
kw['ctr']             = np.where(kw['total_impressions']  > 0,
                                  kw['total_clicks']      / kw['total_impressions'], 0.0)  # diagnostic only

# Match type ordinal encoding (Q5: balanced, specificity-ordered)
kw['match_type_factor'] = kw['match_type'].map({'exact': 1.0, 'phrase': 0.8, 'broad': 0.5})

# Confidence: geometric mean of click saturation and temporal stability
click_sat       = (kw['total_clicks'] / 30).clip(0, 1)
day_sat         = (kw['days_active']  /  7).clip(0, 1)
kw['confidence'] = np.sqrt(click_sat * day_sat)

# ── Stage 2: campaign contextualization ──────────────────────────────────────
camp_stats = kw.groupby('campaign_id').agg(
    _rev   = ('total_revenue', 'sum'),
    _spend = ('total_spend',   'sum'),
).reset_index()
camp_stats['campaign_avg_roas'] = np.where(
    camp_stats['_spend'] > 0,
    camp_stats['_rev'] / camp_stats['_spend'], 0.0
)
kw = kw.merge(camp_stats[['campaign_id', 'campaign_avg_roas']], on='campaign_id')
kw['roas_vs_campaign'] = np.where(
    kw['campaign_avg_roas'] > 0,
    kw['historical_roas'] / kw['campaign_avg_roas'], 0.0
)

print(f"Keyword profiles: {len(kw)}")
kw[['historical_roas', 'cvr', 'confidence', 'roas_vs_campaign']].describe().round(3)

**Feature engineering result:**

`confidence` is near 1.0 for the vast majority of keywords — confirming Q2's finding that the
portfolio is data-rich. `roas_vs_campaign` centred around 1.0 with significant spread, exactly
the differentiation signal the scorer needs.

Keywords with `confidence < 0.3` (sparse) will have their score driven primarily by the LLM
semantic layer built in Section 7.

---
## 7. LLM Semantic Layer — Stage 3

Two components, both grounded in the keyword text rather than click history:

- **Component A** — `all-MiniLM-L6-v2` embeddings (fully offline):
  - A1: centroid distance → specificity score (how long-tail is the keyword?)
  - A2: KNN neighbor ROAS → cold-start prior from real historical data of similar keywords
- **Component B** — Claude `claude-haiku-4-5-20251001` (cached to JSON, committed to repo)

> Architecture plan: **Section 5, Stage 3** in this notebook

### Component A — Embedding Layer (offline)

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

# Encode all keyword texts into 384-dim semantic vectors
embedder    = SentenceTransformer('all-MiniLM-L6-v2')
keywords_list = kw['keyword_text'].tolist()
embeddings  = np.array(embedder.encode(keywords_list, show_progress_bar=True, batch_size=32))

# ── A1: Specificity score — cosine distance from portfolio centroid ───────────
centroid          = embeddings.mean(axis=0, keepdims=True)
sims_to_centroid  = cos_sim(embeddings, centroid).flatten()
kw['llm_specificity'] = 1.0 - sims_to_centroid  # 0 = generic, 1 = highly specific

print("Specificity score (centroid distance):")
print(kw[['keyword_text', 'llm_specificity']].sort_values('llm_specificity', ascending=False).head(5).to_string(index=False))
print("...")
print(kw[['keyword_text', 'llm_specificity']].sort_values('llm_specificity').head(5).to_string(index=False))

In [ ]:
# ── A2: Neighbor ROAS — KNN in embedding space ────────────────────────────────
# For each keyword find its 5 most semantically similar peers and compute
# a similarity-weighted average of their historical ROAS.
# This is the cold-start prior: a sparse keyword inherits real earned ROAS
# from keywords that look like it, not just an abstract semantic score.

K = 5
sim_matrix = cos_sim(embeddings)          # (n, n) pairwise cosine similarities
np.fill_diagonal(sim_matrix, 0.0)         # exclude self-similarity

roas_vals = kw['historical_roas'].values
neighbor_roas = []

for i in range(len(kw)):
    top_k   = np.argsort(sim_matrix[i])[-K:]
    weights = sim_matrix[i][top_k]
    if weights.sum() > 0:
        nr = (weights * roas_vals[top_k]).sum() / weights.sum()
    else:
        nr = roas_vals.mean()
    neighbor_roas.append(nr)

kw['neighbor_roas'] = neighbor_roas

print("Neighbor ROAS sample (sparse keyword perspective):")
sparse = kw[kw['confidence'] < 0.5][['keyword_text', 'historical_roas', 'confidence', 'neighbor_roas']].head(8)
print(sparse.round(3).to_string(index=False))

**Embedding layer result:**

The specificity score correctly separates generic head terms (e.g. "headphones", "laptop") —
score near 0 — from specific long-tail queries — score near 1. This confirms the centroid
distance is a valid specificity proxy for Amazon keywords.

The KNN neighbor ROAS is most meaningful for sparse keywords: a keyword with only 3 days of
history now has a prior anchored to the actual earned ROAS of its closest semantic neighbors,
rather than a cold LLM guess.

### Component B — Claude Structured Metadata (cached)

In [ ]:
import anthropic
import json
import os
from pathlib import Path

CACHE_PATH  = Path('keyword_metadata_cache.json')
MARGIN_MODS = {'commodity': 0.00, 'mid-range': 0.05, 'premium': 0.10}
FUNNEL_MODS = {'awareness': 0.00, 'consideration': 0.05, 'decision': 0.10}
COMP_FLOORS = {'low': 0.20, 'medium': 0.23, 'high': 0.26}

# Load pre-populated cache (committed to repo — reviewers need no API key)
cache: dict = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}

_PROMPT = (
    'Classify this Amazon advertising keyword for bid optimization.\n'
    'Keyword: "{kw}"\n\n'
    'Return ONLY a valid JSON object with exactly these fields:\n'
    '{{"margin_tier": "premium|mid-range|commodity", '
    '"funnel_stage": "awareness|consideration|decision", '
    '"competition_intensity": "low|medium|high"}}'
)
_FALLBACK = {'margin_tier': 'mid-range', 'funnel_stage': 'consideration',
             'competition_intensity': 'medium'}

def fetch_metadata(kw_text: str, client: anthropic.Anthropic) -> dict:
    if kw_text in cache:
        return cache[kw_text]
    try:
        resp   = client.messages.create(
            model='claude-haiku-4-5-20251001', max_tokens=120,
            messages=[{'role': 'user', 'content': _PROMPT.format(kw=kw_text)}]
        )
        result = json.loads(resp.content[0].text.strip())
        assert all(k in result for k in ('margin_tier', 'funnel_stage', 'competition_intensity'))
        cache[kw_text] = result
    except Exception:
        cache[kw_text] = _FALLBACK
    return cache[kw_text]

missing  = [t for t in kw['keyword_text'].unique() if t not in cache]
api_key  = os.environ.get('ANTHROPIC_API_KEY', '')

if missing and api_key:
    client = anthropic.Anthropic(api_key=api_key)
    for kw_text in missing:
        fetch_metadata(kw_text, client)
    CACHE_PATH.write_text(json.dumps(cache, indent=2))
    print(f"Fetched metadata for {len(missing)} keywords — cache updated.")
elif missing:
    for kw_text in missing:
        cache[kw_text] = _FALLBACK
    print(f"No ANTHROPIC_API_KEY — {len(missing)} keywords using neutral fallback.")
else:
    print(f"All {len(cache)} keywords loaded from cache. No API calls needed.")

# Apply metadata to dataframe
kw['llm_margin_mod']  = kw['keyword_text'].map(
    lambda t: MARGIN_MODS.get(cache.get(t, _FALLBACK)['margin_tier'],   0.05))
kw['llm_funnel_mod']  = kw['keyword_text'].map(
    lambda t: FUNNEL_MODS.get(cache.get(t, _FALLBACK)['funnel_stage'],  0.05))
kw['floor_bid_comp']  = kw['keyword_text'].map(
    lambda t: COMP_FLOORS.get(cache.get(t, _FALLBACK)['competition_intensity'], 0.23))

print("\nSample metadata:")
sample_meta = pd.DataFrame([
    {'keyword': t, **cache.get(t, _FALLBACK)}
    for t in kw['keyword_text'].sample(8, random_state=42)
])
print(sample_meta.to_string(index=False))

**LLM metadata result:**

Claude correctly identifies premium/decision keywords (specific branded queries with high
purchase intent) versus commodity/awareness keywords (generic category searches). The
`competition_intensity` field will feed the bid floor in Stage 5 — not the score — because
competition affects the minimum bid needed to enter the auction, not the keyword's efficiency.

---
## 8. Scoring & Portfolio Optimization — Stages 4 & 5

Stage 4 combines all features into a single keyword score. Stage 5 derives the optimal bid
from that score, enforces all constraints, and runs the asymmetric budget slash if needed.

> Architecture plan: **Section 5, Stages 4 & 5** in this notebook

In [ ]:
from scipy.stats import rankdata

def pct_rank(series: pd.Series) -> pd.Series:
    """Percentile-rank normalisation → [0, 1]. Robust to extreme outliers (Q4)."""
    r = rankdata(series.fillna(0).values, method='average')
    return pd.Series((r - 1) / max(len(r) - 1, 1), index=series.index)

# Percentile-rank normalise the three behavioral inputs
kw['roas_vs_campaign_norm'] = pct_rank(kw['roas_vs_campaign'])
kw['cvr_norm']              = pct_rank(kw['cvr'])
kw['neighbor_roas_norm']    = pct_rank(kw['neighbor_roas'])

# Behavioral score — weighted additive sum (Stage 4)
kw['behavioral_score'] = (
    0.60 * kw['roas_vs_campaign_norm'] +
    0.25 * kw['cvr_norm'] +
    0.15 * kw['match_type_factor']
)

# Semantic prior — used when confidence is low (sparse keywords)
kw['semantic_prior'] = (
    0.60 * kw['neighbor_roas_norm'] +
    0.40 * kw['llm_specificity']
)

# Unified keyword score — confidence-weighted blend + LLM modifiers
kw['keyword_score'] = (
    kw['confidence']       * kw['behavioral_score'] +
    (1 - kw['confidence']) * kw['semantic_prior'] +
    kw['llm_margin_mod'] +
    kw['llm_funnel_mod']
)

print("Keyword score distribution:")
print(kw['keyword_score'].describe().round(3))
print()

# Show top and bottom 5 keywords by score
top5 = kw.nlargest(5,  'keyword_score')[['keyword_text', 'keyword_score', 'historical_roas', 'confidence']]
bot5 = kw.nsmallest(5, 'keyword_score')[['keyword_text', 'keyword_score', 'historical_roas', 'confidence']]
print("Top 5 keywords by score:")
print(top5.round(3).to_string(index=False))
print("\nBottom 5 keywords by score:")
print(bot5.round(3).to_string(index=False))

**Scoring result:**

High-score keywords show a consistent pattern: strong relative ROAS within their campaign,
healthy CVR, and exact/phrase match type. Low-score keywords typically have sub-campaign ROAS
and broad match — the expected result given the feature weights.

The confidence field is near 1.0 for almost the entire portfolio (as Q2 predicted), meaning
the semantic prior is barely used for most keywords. The few truly sparse keywords have their
score grounded in neighbor ROAS rather than raw zero-history defaulting.

In [ ]:
def optimize_campaign(camp_df: pd.DataFrame) -> pd.DataFrame:
    df     = camp_df.copy()
    budget = df['campaign_daily_budget'].iloc[0]
    total_score = df['keyword_score'].sum()

    # Step 1 — Score-to-bid: each keyword's budget share → per-click bid
    df['expected_clicks'] = df['avg_daily_clicks'].clip(lower=0.5)  # floor: avoid /0
    df['raw_bid'] = (
        (df['keyword_score'] / total_score) * budget / df['expected_clicks']
    )

    # Step 2 — Stability cap ±50% (applied after bid derivation)
    df['raw_bid'] = df['raw_bid'].clip(
        lower = df['current_avg_bid'] * 0.50,
        upper = df['current_avg_bid'] * 1.50,
    )

    # Step 3 — Competition bid floor modifier
    df['floor_bid'] = df['floor_bid_comp'].clip(lower=BID_MIN)
    df['raw_bid']   = df[['raw_bid', 'floor_bid']].max(axis=1)

    # Step 4 — Asymmetric safety slash (safety net for rounding / click variance)
    df['projected_spend'] = df['raw_bid'] * df['expected_clicks']

    if df['projected_spend'].sum() > budget + 0.01:
        df = df.sort_values('keyword_score').copy()  # worst keyword first

        for idx in df.index:
            if df['projected_spend'].sum() <= budget + 0.01:
                break
            excess   = df['projected_spend'].sum() - budget
            bid      = df.loc[idx, 'raw_bid']
            floor    = df.loc[idx, 'floor_bid']
            clicks   = df.loc[idx, 'expected_clicks']
            headroom = (bid - floor) * clicks  # max reducible spend for this keyword

            if headroom <= 0:
                continue  # already at floor, skip

            cut = min(excess, headroom)
            df.loc[idx, 'raw_bid']          = bid - cut / clicks
            df.loc[idx, 'projected_spend']  = df.loc[idx, 'raw_bid'] * clicks

        # Safety net: if all keywords at floor and still over, proportional scale-down
        total_proj = df['projected_spend'].sum()
        if total_proj > budget + 0.01:
            scale         = budget / total_proj
            df['raw_bid'] = (df['raw_bid'] * scale).clip(lower=df['floor_bid'])
            df['projected_spend'] = df['raw_bid'] * df['expected_clicks']
            df['_budget_flag'] = True  # campaign flagged

    return df

results = [optimize_campaign(g) for _, g in kw.groupby('campaign_id')]
kw_opt  = pd.concat(results).reset_index(drop=True)

**Portfolio optimization result:**

In [ ]:
# Verify budget compliance per campaign
budget_check = kw_opt.groupby(['campaign_name', 'campaign_daily_budget']).agg(
    projected_spend = ('projected_spend', 'sum'),
    n_keywords      = ('keyword_id',      'count'),
).reset_index()
budget_check['utilisation']    = budget_check['projected_spend'] / budget_check['campaign_daily_budget']
budget_check['within_budget']  = budget_check['projected_spend'] <= budget_check['campaign_daily_budget'] + 0.01

print("Campaign budget compliance after optimization:")
print(budget_check[['campaign_name', 'campaign_daily_budget',
                     'projected_spend', 'utilisation', 'within_budget']].round(2).to_string(index=False))
print()
print(f"Campaigns within budget: {budget_check['within_budget'].sum()} / {len(budget_check)}")

---
## 9. Output — Stage 6: Marketplace Compliance & Recommendations

Final ceiling clip (`min(bid, $15.00)`) — the $0.20 floor is already guaranteed by Stage 5's
loop. Then generate a human-readable `reason_or_score` column per keyword and write
`bid_recommendations.csv`.

> Architecture plan: **Section 5, Stage 6** in this notebook

In [ ]:
# Stage 6: ceiling clip only (floor already enforced in Stage 5)
kw_opt['recommended_bid'] = kw_opt['raw_bid'].clip(upper=BID_MAX)

_MARGIN_LABELS = {0.00: 'commodity', 0.05: 'mid-range', 0.10: 'premium'}
_FUNNEL_LABELS = {0.00: 'awareness', 0.05: 'consideration', 0.10: 'decision'}

def build_reason(row) -> str:
    parts = []

    # Primary score driver
    conf = row['confidence']
    if conf >= 0.8:
        parts.append(
            f"behavioral: ROAS {row['historical_roas']:.2f}x "
            f"(campaign avg {row['campaign_avg_roas']:.2f}x, "
            f"relative {row['roas_vs_campaign']:.2f}x)"
        )
    elif conf <= 0.2:
        parts.append(
            f"LLM-primary (confidence={conf:.2f}): "
            f"neighbor ROAS prior {row['neighbor_roas']:.2f}x"
        )
    else:
        parts.append(
            f"blended (confidence={conf:.2f}): "
            f"ROAS {row['historical_roas']:.2f}x + "
            f"neighbor prior {row['neighbor_roas']:.2f}x"
        )

    # LLM metadata summary
    parts.append(
        f"margin={_MARGIN_LABELS.get(row['llm_margin_mod'], 'mid-range')}, "
        f"funnel={_FUNNEL_LABELS.get(row['llm_funnel_mod'], 'consideration')}"
    )

    # Bid change narrative
    change_pct = (row['recommended_bid'] / row['current_avg_bid'] - 1) * 100
    if row['recommended_bid'] <= row['floor_bid'] + 0.01:
        parts.append("bid at marketplace floor (budget constraint)")
    elif change_pct >=  40:
        parts.append(f"bid raised +{change_pct:.0f}% (high efficiency, under-invested)")
    elif change_pct <= -40:
        parts.append(f"bid cut {change_pct:.0f}% (low efficiency, over-invested)")

    return "; ".join(parts)

kw_opt['reason_or_score'] = kw_opt.apply(build_reason, axis=1)

output = kw_opt[['keyword_id', 'keyword_text', 'current_avg_bid',
                  'recommended_bid', 'reason_or_score']].copy()
output.to_csv('bid_recommendations.csv', index=False)
print(f"Saved {len(output)} recommendations to bid_recommendations.csv")
print()
print("Sample output:")
print(output.head(8).to_string(index=False))

**Final output:**

`bid_recommendations.csv` contains one row per keyword with:
- `recommended_bid` — the optimised CPC bid, compliant with all marketplace and budget constraints
- `reason_or_score` — a human-readable explanation of the key signals and budget action taken

Every recommendation traces back to a specific EDA finding:
- Bids derived from portfolio score allocation (Q6: all campaigns over-budget)
- ROAS normalised within campaign (Q4: extreme variance)
- Aggregated over active days only (Q1: informative absence)
- Sparse keywords backed by neighbor ROAS prior (Q2: 3% zero-conversion keywords)
- No temporal weighting (Q7: no significant trend)

---
## 10. Operational Validation & Backtesting Framework

We cannot use a temporal train/test split here — explained in Section 5.0. Instead we
validate through three operational checks that directly answer the only question that matters:
**did the algorithm satisfy every constraint AND improve ROAS?**

1. **Hard boundary assertions** — bids within marketplace bounds and stability cap
2. **Budget feasibility** — every campaign's projected spend sits below its cap
3. **ROAS improvement** — predicted portfolio ROAS exceeds historical baseline

> Full methodology and limitations documented in `README.md § Operational Validation`

### 10.1 — Hard Boundary & Stability Assertions

In [ ]:
# All assertions use kw_opt produced in Section 8/9
rb  = kw_opt['recommended_bid']
cab = kw_opt['current_avg_bid']

# ── Marketplace bounds ────────────────────────────────────────────────────────
assert rb.min() >= 0.20,  f"FAIL: bid below $0.20 floor — min={rb.min():.4f}"
assert rb.max() <= 15.00, f"FAIL: bid above $15.00 ceiling — max={rb.max():.4f}"
print(f"Marketplace bounds PASSED")
print(f"  min recommended_bid = ${rb.min():.2f}  (floor $0.20)")
print(f"  max recommended_bid = ${rb.max():.2f}  (ceiling $15.00)")
print()

# ── Stability cap ±50% ────────────────────────────────────────────────────────
lower = cab * 0.50
upper = cab * 1.50
violates_upper = (rb > upper + 0.01)
# Lower-bound violations are expected when budget forces keywords to their
# marketplace floor ($0.20-$0.26) — that floor takes priority over the cap.
violates_lower_true = (rb < lower - 0.01) & (rb > kw_opt['floor_bid'] + 0.01)

assert not violates_upper.any(),     f"FAIL: {violates_upper.sum()} keywords exceed +50% stability cap"
assert not violates_lower_true.any(),     f"FAIL: {violates_lower_true.sum()} keywords breach -50% cap without floor justification"

floor_forced = (rb < lower - 0.01) & (rb <= kw_opt['floor_bid'] + 0.01)
print(f"Stability cap (±50%) PASSED")
print(f"  Upper-cap violations: 0")
print(f"  Lower-cap violations: 0 (true)")
if floor_forced.sum() > 0:
    print(f"  Floor-forced reductions: {floor_forced.sum()} keywords "
          f"(budget constraint drove bid to marketplace floor — expected behaviour)")
print()
print("All hard constraints satisfied.")

**Result:** Every bid is within [$0.20, $15.00]. No keyword exceeds the +50% stability cap.
Keywords whose bid falls below the -50% lower bound were forced there by the budget constraint
driving them to the marketplace floor — this is intentional and correct: the budget constraint
takes priority over the stability cap when a campaign has no other way to reach compliance.

### 10.2 — Campaign Budget Feasibility

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Historical avg daily spend per campaign (from raw data)
hist_spend = df.groupby('campaign_name').agg(
    historical_avg_daily = ('spend', lambda x: x.groupby(df.loc[x.index, 'date']).sum().mean())
).reset_index()

# Projected optimized daily spend per campaign
proj_spend = kw_opt.groupby(['campaign_name', 'campaign_daily_budget']).apply(
    lambda g: (g['recommended_bid'] * g['expected_clicks']).sum()
).reset_index(name='projected_daily_spend')

merged = proj_spend.merge(hist_spend, on='campaign_name').sort_values('campaign_name')

x      = np.arange(len(merged))
width  = 0.30
labels = [name.replace(' & ', '
& ') for name in merged['campaign_name']]

fig, ax = plt.subplots(figsize=(14, 6))

bars_hist = ax.bar(x - width/2, merged['historical_avg_daily'],
                   width, label='Historical avg daily spend', color='#e07b54', alpha=0.85)
bars_proj = ax.bar(x + width/2, merged['projected_daily_spend'],
                   width, label='Projected optimized spend', color='#4a90d9', alpha=0.85)

# Budget cap line per campaign
for i, (_, row) in enumerate(merged.iterrows()):
    ax.plot([i - width, i + width], [row['campaign_daily_budget']] * 2,
            color='red', linewidth=2.5, linestyle='--', zorder=5)

budget_line = mpatches.Patch(color='red', linestyle='--', label='Daily budget cap')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Daily Spend ($)')
ax.set_title('Campaign Budget: Historical Overspend vs Optimized Projection
'
             'Every optimized bar sits below its red budget cap line', fontsize=13)
ax.legend(handles=[bars_hist, bars_proj, budget_line], fontsize=10)
ax.yaxis.grid(True, linestyle='--', alpha=0.5)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

print("Budget compliance summary:")
for _, row in merged.iterrows():
    status = "WITHIN" if row['projected_daily_spend'] <= row['campaign_daily_budget'] else "OVER"
    hist_ratio = row['historical_avg_daily'] / row['campaign_daily_budget']
    proj_ratio = row['projected_daily_spend'] / row['campaign_daily_budget']
    print(f"  {row['campaign_name']:<28} "
          f"was {hist_ratio:.1f}x budget  →  now {proj_ratio:.2f}x  [{status}]")

**Result:** Every campaign's projected spend now sits below its daily budget cap.
Previously every campaign was 1.6×–4.8× over budget (Q6). The asymmetric slash concentrated
cuts on the lowest-score keywords, bringing each campaign into compliance while preserving
the bids of high-ROAS keywords.

### 10.3 — Counterfactual ROAS Improvement Proof

We cannot run a live A/B test, so we use an offline economic proxy:
project what revenue the *recommended* bids would generate, assuming each keyword's
revenue-per-click stays constant but applies a mild **elasticity decay** `(bid_new/bid_old)^0.1`
— a diminishing-returns penalty that makes the simulation conservative: raising a bid by 50%
only increases revenue efficiency by ~4%, not 50%.

In [ ]:
# ── Baseline: actual historical portfolio ROAS ────────────────────────────────
actual_roas = kw_opt['total_revenue'].sum() / kw_opt['total_spend'].sum()

# ── Counterfactual: projected next-cycle metrics ──────────────────────────────
# revenue_per_click = what each click historically returned in revenue
kw_opt['revenue_per_click'] = kw_opt['total_revenue'] / (kw_opt['total_clicks'] + 1e-5)

# Elasticity decay: mild diminishing-returns penalty on bid changes
elasticity = (kw_opt['recommended_bid'] / (kw_opt['current_avg_bid'] + 1e-5)) ** 0.1

kw_opt['predicted_spend_30d']   = kw_opt['recommended_bid'] * kw_opt['avg_daily_clicks'] * 30
kw_opt['predicted_revenue_30d'] = (
    kw_opt['predicted_spend_30d'] * kw_opt['revenue_per_click'] * elasticity
)

predicted_roas = kw_opt['predicted_revenue_30d'].sum() / kw_opt['predicted_spend_30d'].sum()

# ── Portfolio-level result ────────────────────────────────────────────────────
print("=" * 52)
print(f"  Actual historical portfolio ROAS:   {actual_roas:.3f}x")
print(f"  Predicted optimized portfolio ROAS: {predicted_roas:.3f}x")
print(f"  Improvement:                       +{(predicted_roas/actual_roas - 1)*100:.1f}%")
print("=" * 52)
print()
if predicted_roas > actual_roas:
    print("Predicted ROAS > Actual ROAS: CONFIRMED")
else:
    print("WARNING: predicted ROAS did not improve — review scoring weights")
print()

# ── Per-campaign breakdown ────────────────────────────────────────────────────
camp_val = kw_opt.groupby('campaign_name').apply(lambda g: pd.Series({
    'actual_roas':    g['total_revenue'].sum() / (g['total_spend'].sum() + 1e-5),
    'predicted_roas': g['predicted_revenue_30d'].sum() / (g['predicted_spend_30d'].sum() + 1e-5),
})).reset_index()
camp_val['improvement_pct'] = (camp_val['predicted_roas'] / camp_val['actual_roas'] - 1) * 100
camp_val['better'] = camp_val['predicted_roas'] > camp_val['actual_roas']

print("Per-campaign ROAS (actual vs predicted):")
print(f"{'Campaign':<28} {'Actual':>8} {'Predicted':>10} {'Change':>8}  {'':>6}")
for _, r in camp_val.sort_values('improvement_pct', ascending=False).iterrows():
    arrow = '↑' if r['better'] else '↓'
    print(f"  {r['campaign_name']:<26} {r['actual_roas']:>7.2f}x "
          f"{r['predicted_roas']:>9.2f}x  {r['improvement_pct']:>+6.1f}%  {arrow}")
print()
print(f"Campaigns with improved ROAS: {camp_val['better'].sum()} / {len(camp_val)}")

**Result:** Predicted portfolio ROAS exceeds the historical baseline.

The mechanism is straightforward: the algorithm identified which keywords were consuming budget
below campaign-average ROAS and cut their bids toward the $0.20 floor. The freed budget was
reallocated proportionally to high-score keywords. Even with the conservative elasticity decay
penalty, concentrating spend on efficient keywords raises the portfolio ROAS.

**Limitations of this simulation:**
- Revenue-per-click is assumed constant at historical levels — real auctions are dynamic
- The elasticity exponent (0.1) is a conservative approximation, not a fitted bid-response curve
- This is an offline proxy, not a live A/B test; it cannot capture competitor bid reactions

These limitations are inherent to any offline bid optimizer. The simulation is intentionally
conservative (diminishing returns penalise bid increases) to avoid overstating the improvement.